# 建立中文 RAG 知識庫

## 1. 安裝所需套件

In [ ]:
%pip install -q "sentence-transformers==6.1.0" "transformers==5.20.1" "faiss-cpu==1.15.1"

## 2. 檢查環境與集中設定

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import importlib.metadata as package_metadata
import hashlib
import json
import re
import tempfile
import zipfile

import torch
import faiss
from IPython.display import display
from sentence_transformers import SentenceTransformer
from huggingface_hub import HfApi

In [ ]:
MODEL_ID = "google/embeddinggemma-2"
BATCH_SIZE = 4
EXPECTED_DOCUMENT_COUNT = None  # None 不強制數量
EXPORT_ZIP_NAME = "vector_embedding.zip"

# text 本身已含標題；這裡不再從 metadata 添加標題。
DOCUMENT_TEMPLATE = "title: none | text: {text}"
QUERY_TEMPLATE = "task: search result | query: {question}"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = (
    torch.bfloat16
    if DEVICE == "cuda" and torch.cuda.is_bf16_supported()
    else torch.float32
)
PACKAGE_NAMES = ["torch", "numpy", "sentence-transformers", "transformers", "faiss-cpu", "huggingface-hub"]
PACKAGE_VERSIONS = {name: package_metadata.version(name) for name in PACKAGE_NAMES}
print("裝置：", DEVICE, "／模型精度：", DTYPE)
if DEVICE == "cuda":
    print("GPU：", torch.cuda.get_device_name(0))
display(pd.DataFrame(PACKAGE_VERSIONS.items(), columns=["套件", "版本"]))

裝置： cuda ／模型精度： torch.bfloat16
GPU： Tesla T4


,套件,版本
0,torch,2.11.0+cu130
1,numpy,2.1.3
2,sentence-transformers,6.1.0
3,transformers,5.20.1
4,faiss-cpu,1.15.1
5,huggingface-hub,1.33.0


## 3. 從 github 讀取 dataset ZIP

In [ ]:
from urllib.request import urlopen

# 我的 Github 檔案連結
GITHUB_URL = "https://raw.githubusercontent.com/Cornnnnnnnnn/Stardew-Valley-RAG/main/stardew_valley_dataset.zip"

archive_name = "stardew_valley_dataset.zip"

with urlopen(GITHUB_URL, timeout=60) as response:
    archive_bytes = response.read()

print(f"檔案大小：{len(archive_bytes):,} bytes")

SOURCE_ARCHIVE_SHA256 = hashlib.sha256(archive_bytes).hexdigest()
DATA_DIR = Path(tempfile.mkdtemp(prefix="rag_documents_"))

# 僅解開 TXT，並確保 ZIP 中的路徑不會寫到暫存資料夾外。
import io
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    members = [
        member for member in archive.infolist()
        if not member.is_dir()
        and member.filename.lower().endswith(".txt")
        and not any(part.startswith((".", "__MACOSX")) for part in Path(member.filename).parts)
    ]
    if not members:
        raise ValueError("ZIP 中找不到 TXT；請確認使用第一版教材，或準備自己的 TXT。")
    destinations = set()
    for member in members:
        target = (DATA_DIR / member.filename).resolve()
        if not target.is_relative_to(DATA_DIR.resolve()):
            raise ValueError(f"ZIP 包含不適當的檔案路徑：{member.filename}")
        if target in destinations:
            raise ValueError(f"ZIP 有重複檔名：{member.filename}")
        destinations.add(target)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(member))

txt_paths = sorted(DATA_DIR.rglob("*.txt"), key=lambda p: p.relative_to(DATA_DIR).as_posix())
txt_paths = sorted(
    [p for p in DATA_DIR.rglob("*") if p.is_file() and p.suffix.lower() == ".txt"],
    key=lambda p: p.relative_to(DATA_DIR).as_posix(),
)
print("資料集名稱：", archive_name)
print("TXT 數量：", len(txt_paths))
for path in txt_paths:
    print("•", path.relative_to(DATA_DIR).as_posix())
if EXPECTED_DOCUMENT_COUNT is not None and len(txt_paths) != EXPECTED_DOCUMENT_COUNT:
    print(f"提醒：本次為 {len(txt_paths)} 篇，與原教材的 {EXPECTED_DOCUMENT_COUNT} 篇不同；仍可繼續。")

檔案大小：21,171 bytes
資料集名稱： stardew_valley_dataset.zip
TXT 數量： 24
• 01_新手指南.txt
• 02_遊戲內時間.txt
• 03_技能.txt
• 04_季節.txt
• 05_農作物.txt
• 06_動物.txt
• 07_雞舍.txt
• 08_農舍.txt
• 09_溫室.txt
• 10_工匠物品.txt
• 11_工具.txt
• 12_釣魚.txt
• 13_魚.txt
• 14_礦井.txt
• 15_武器.txt
• 16_採集.txt
• 17_居民.txt
• 18_友誼.txt
• 19_任務.txt
• 20_收集包.txt
• 21_社區中心.txt
• 22_烹飪.txt
• 23_節日.txt
• 24_薑島.txt


## 4. 看第一篇原始文件

In [ ]:
SAMPLE_INDEX = 0
sample_path = txt_paths[SAMPLE_INDEX]
try:
    sample_raw = sample_path.read_text(encoding="utf-8-sig")
except UnicodeDecodeError as exc:
    raise ValueError(f"{sample_path.name} 不是可讀取的 UTF-8 文字。") from exc
print("檔名：", sample_path.name)
print("=" * 50)
print(sample_raw)

檔名： 01_新手指南.txt
標題：新手指南
來源網址：https://zh.stardewvalleywiki.com/新手指南
教材整理日期：2026-11-02
【內容】
《星露谷物語》是開放式農場生活遊戲。新玩家除了種田，也能釣魚、採集、採礦、戰鬥、製作物品、烹飪、交朋友、完成任務與修復社區中心。開始時可決定角色外觀與農場地圖，不同農場適合不同玩法，選定後一般無法更換。
一般開局可以先清出一小塊耕地，用鋤頭翻土，種下取得的防風草種子（部分農場開局獎勵不同），再用澆水壺每日澆水。防風草約四天後成熟。農作物缺水時通常停止生長而不是立即枯死。
留意體力與時間：吃食物能補充體力，睡覺可以進入下一天。電視能提供天氣預報、運氣與料理等資訊；新手不必只靠種田獲利，可依興趣發展不同技能。



## 5. 取出 RAG 文字

In [ ]:
def extract_rag_text(raw_text):
    """從本教材 TXT 取出標題與正文；換資料時修改這裡。"""
    normalized = raw_text.lstrip("\ufeff").replace("\r\n", "\n").replace("\r", "\n")
    title_match = re.search(r"^標題[：:]\s*(.+)$", normalized, flags=re.MULTILINE)
    content_match = re.search(
        r"^【內容】[ \t]*\n(?P<body>.*?)(?=^【[^】\n]+】|\Z)",
        normalized,
        flags=re.MULTILINE | re.DOTALL,
    )
    if content_match is None or not content_match.group("body").strip():
        raise ValueError("找不到非空的【內容】正文；請修改 extract_rag_text() 以配合文件格式。")
    title = title_match.group(1).strip() if title_match else ""
    body = content_match.group("body").strip()
    return f"{title}\n\n{body}" if title else body

sample_rag_text = extract_rag_text(sample_raw)
print(sample_rag_text)
print("\n擷取後字元數：", len(sample_rag_text))

新手指南

《星露谷物語》是開放式農場生活遊戲。新玩家除了種田，也能釣魚、採集、採礦、戰鬥、製作物品、烹飪、交朋友、完成任務與修復社區中心。開始時可決定角色外觀與農場地圖，不同農場適合不同玩法，選定後一般無法更換。
一般開局可以先清出一小塊耕地，用鋤頭翻土，種下取得的防風草種子（部分農場開局獎勵不同），再用澆水壺每日澆水。防風草約四天後成熟。農作物缺水時通常停止生長而不是立即枯死。
留意體力與時間：吃食物能補充體力，睡覺可以進入下一天。電視能提供天氣預報、運氣與料理等資訊；新手不必只靠種田獲利，可依興趣發展不同技能。

擷取後字元數： 261


## 6. 取出 metadata

In [ ]:
def extract_metadata(raw_text, filename):
    """取出本教材的描述資訊；換資料時修改這裡。"""
    normalized = raw_text.lstrip("\ufeff").replace("\r\n", "\n").replace("\r", "\n")
    def read_field(name):
        match = re.search(rf"^{re.escape(name)}[：:][ \t]*(.*)$", normalized, flags=re.MULTILINE)
        return match.group(1).strip() or None if match else None

    return {
        "title": read_field("標題") or Path(filename).stem,
        "source_url": read_field("來源網址"),
        "date": read_field("教材整理日期"),
        "filename": filename,
    }

sample_metadata = extract_metadata(sample_raw, sample_path.relative_to(DATA_DIR).as_posix())
print(json.dumps(sample_metadata, ensure_ascii=False, indent=2))

{
  "title": "新手指南",
  "source_url": "https://zh.stardewvalleywiki.com/新手指南",
  "date": "2026-11-02",
  "filename": "01_新手指南.txt"
}


## 7. 批次整理成統一格式

In [ ]:
documents = []
for number, path in enumerate(txt_paths, start=1):
    filename = path.relative_to(DATA_DIR).as_posix()
    try:
        raw_text = path.read_text(encoding="utf-8-sig")
        if "\ufffd" in raw_text:
            raise ValueError("文字含有替代字元，請檢查原始編碼。")
        rag_text = extract_rag_text(raw_text)
        metadata = extract_metadata(raw_text, filename)
    except (ValueError, UnicodeDecodeError) as exc:
        raise ValueError(f"處理失敗：{filename}；原因：{exc}") from exc
    documents.append({"doc_id": f"doc_{number:03d}", "text": rag_text, "metadata": metadata})

if not documents:
    raise ValueError("沒有可建立索引的文件。")
index_doc_ids = [doc["doc_id"] for doc in documents]
if len(set(index_doc_ids)) != len(index_doc_ids):
    raise ValueError("doc_id 重複，請檢查編號規則。")

summary = pd.DataFrame([
    {"doc_id": doc["doc_id"], "標題": doc["metadata"]["title"],
     "正文含標題字元數": len(doc["text"]), "來源": doc["metadata"]["source_url"],
     "整理日期": doc["metadata"]["date"]}
    for doc in documents
])
display(summary)
for doc in documents:
    missing = [key for key in ("source_url", "date") if not doc["metadata"].get(key)]
    if missing:
        print("提醒：", doc["doc_id"], "缺少", ", ".join(missing))
print("第一筆統一格式：")
print(json.dumps(documents[0], ensure_ascii=False, indent=2))

,doc_id,標題,正文含標題字元數,來源,整理日期
0,doc_001,新手指南,261,https://zh.stardewvalleywiki.com/新手指南,2026-11-02
1,doc_002,遊戲內時間,241,https://zh.stardewvalleywiki.com/游戏内时间,2026-11-02
2,doc_003,技能與職業,250,https://zh.stardewvalleywiki.com/技能,2026-11-02
3,doc_004,四季與換季,228,https://zh.stardewvalleywiki.com/季节,2026-11-02
4,doc_005,農作物與種植,266,https://zh.stardewvalleywiki.com/农作物,2026-11-02
5,doc_006,動物與畜牧,245,https://zh.stardewvalleywiki.com/动物,2026-11-02
6,doc_007,雞舍與升級,283,https://zh.stardewvalleywiki.com/鸡舍,2026-11-02
7,doc_008,農舍與房屋升級,256,https://zh.stardewvalleywiki.com/农舍,2026-11-02
8,doc_009,溫室與全年種植,247,https://zh.stardewvalleywiki.com/温室,2026-11-02
9,doc_010,工匠物品與加工,246,https://zh.stardewvalleywiki.com/工匠物品,2026-11-02


第一筆統一格式：
{
  "doc_id": "doc_001",
  "text": "新手指南\n\n《星露谷物語》是開放式農場生活遊戲。新玩家除了種田，也能釣魚、採集、採礦、戰鬥、製作物品、烹飪、交朋友、完成任務與修復社區中心。開始時可決定角色外觀與農場地圖，不同農場適合不同玩法，選定後一般無法更換。\n一般開局可以先清出一小塊耕地，用鋤頭翻土，種下取得的防風草種子（部分農場開局獎勵不同），再用澆水壺每日澆水。防風草約四天後成熟。農作物缺水時通常停止生長而不是立即枯死。\n留意體力與時間：吃食物能補充體力，睡覺可以進入下一天。電視能提供天氣預報、運氣與料理等資訊；新手不必只靠種田獲利，可依興趣發展不同技能。",
  "metadata": {
    "title": "新手指南",
    "source_url": "https://zh.stardewvalleywiki.com/新手指南",
    "date": "2026-11-02",
    "filename": "01_新手指南.txt"
  }
}


## 8. 載入 EmbeddingGemma 2

In [ ]:
try:
    MODEL_REVISION = HfApi().model_info(MODEL_ID, token=False).sha
    model = SentenceTransformer(
        MODEL_ID,
        revision=MODEL_REVISION,
        token=False,
        device=DEVICE,
        config_kwargs={"vision_config": None, "audio_config": None},
        model_kwargs={"dtype": DTYPE, "attn_implementation": "eager"},
    )
except Exception as exc:
    raise RuntimeError(
        "EmbeddingGemma 2 載入失敗。請確認網路、套件版本，必要時重新啟動工作階段；"
        "若 GPU 記憶體不足，改用 CPU。未替換成其他模型。"
    ) from exc

print("模型：", MODEL_ID)
print("版本：", MODEL_REVISION)
print("裝置：", model.device)
print("模型精度：", model.dtype)
print("向量維度：", model.get_sentence_embedding_dimension())
print("模型單筆 token 上限：", model.max_seq_length)

modules.json:   0%|          | 0.00/413 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/1.56k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/22.2k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.46k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.49GB            

... (8 outputs left out)

config.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/97.0 [00:00<?, ?B/s]

模型： google/embeddinggemma-2
版本： f9972ac22e696a105c482202646979e33e4b5a1e
裝置： cuda:0
模型精度： torch.bfloat16
向量維度： 768
模型單筆 token 上限： 1000000000000000019884624838656


/tmp/ipykernel_1997/465943176.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("向量維度：", model.get_sentence_embedding_dimension())


## 9. 檢查輸入長度

Note: 在進行向量化之前，先利用 Tokenizer 計算每篇文章的 Token 數量，並檢查是否超過模型的輸入長度限制

In [ ]:
document_inputs = [DOCUMENT_TEMPLATE.format(text=doc["text"]) for doc in documents]
TOKEN_LIMIT = min(2048, int(model.max_seq_length))

if TOKEN_LIMIT <= 0:
    raise ValueError("輸入長度上限必須大於0。")

def count_input_tokens(text):
    return len(model.tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"])

token_counts = [count_input_tokens(text) for text in document_inputs]
length_table = pd.DataFrame({"doc_id": index_doc_ids, "tokens_含前綴": token_counts})
display(length_table)
long_docs = [doc_id for doc_id, length in zip(index_doc_ids, token_counts) if length > TOKEN_LIMIT]
if long_docs:
    raise ValueError(f"以下文件超過 {TOKEN_LIMIT} tokens，請先調整資料：{long_docs}")
print("本教材使用的 token 上限：", TOKEN_LIMIT)
print("全部文件均未超過上限；不需要截斷。")
print("第一筆真正送入 embedding 的文字：\n", document_inputs[0])





,doc_id,tokens_含前綴
0,doc_001,211
1,doc_002,183
2,doc_003,201
3,doc_004,184
4,doc_005,204
5,doc_006,192
6,doc_007,248
7,doc_008,202
8,doc_009,207
9,doc_010,191


本教材使用的 token 上限： 2048
全部文件均未超過上限；不需要截斷。
第一筆真正送入 embedding 的文字：
 title: none | text: 新手指南

《星露谷物語》是開放式農場生活遊戲。新玩家除了種田，也能釣魚、採集、採礦、戰鬥、製作物品、烹飪、交朋友、完成任務與修復社區中心。開始時可決定角色外觀與農場地圖，不同農場適合不同玩法，選定後一般無法更換。
一般開局可以先清出一小塊耕地，用鋤頭翻土，種下取得的防風草種子（部分農場開局獎勵不同），再用澆水壺每日澆水。防風草約四天後成熟。農作物缺水時通常停止生長而不是立即枯死。
留意體力與時間：吃食物能補充體力，睡覺可以進入下一天。電視能提供天氣預報、運氣與料理等資訊；新手不必只靠種田獲利，可依興趣發展不同技能。


## 10. 產生並觀察文件向量

Note: 將 24 篇 Stardew Valley 文章轉換成數值向量，每篇文章對應向量矩陣中的一列


In [ ]:
embeddings = model.encode(
    document_inputs,
    prompt="",
    batch_size=BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
embeddings = np.ascontiguousarray(embeddings, dtype=np.float32)
if embeddings.ndim != 2 or embeddings.shape[0] != len(documents):
    raise ValueError("向量形狀與文件數不符。")
if not np.isfinite(embeddings).all():
    raise ValueError("向量含有 NaN 或無限值。")
if np.any(np.linalg.norm(embeddings, axis=1) <= 0):
    raise ValueError("出現零向量。")
faiss.normalize_L2(embeddings)
DIMENSION = int(embeddings.shape[1])
if DIMENSION != int(model.get_sentence_embedding_dimension()):
    raise ValueError("實際向量維度與模型宣告不符。")
print("矩陣形狀（文件數, 向量維度）：", embeddings.shape)
print("第一篇向量的前10個數字：", embeddings[0, :10])
print("前5篇向量長度：", np.linalg.norm(embeddings[:5], axis=1))

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

矩陣形狀（文件數, 向量維度）： (24, 768)
第一篇向量的前10個數字： [ 0.0191028   0.05061632 -0.02688992 -0.00784796  0.03358199  0.0479395
  0.03005344  0.00736127  0.01015977 -0.13043438]
前5篇向量長度： [1.        1.        1.0000001 1.0000001 1.0000001]


/tmp/ipykernel_1997/2199939656.py:18: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  if DIMENSION != int(model.get_sentence_embedding_dimension()):


## 11. 建立 FAISS 精確搜尋索引

Note: 由於向量已經過正規化，因此可以透過內積計算 Cosine Similarity，找出與使用者問題最相關的文章

In [ ]:
index = faiss.IndexFlatIP(DIMENSION)
index.add(embeddings)
documents_by_id = {doc["doc_id"]: doc for doc in documents}
assert index.ntotal == len(index_doc_ids) == len(documents)
print("索引類型：", type(index).__name__)
print("索引向量數：", index.ntotal)
print("索引維度：", index.d)
display(pd.DataFrame({"FAISS編號": range(len(index_doc_ids)), "doc_id": index_doc_ids}))

索引類型： IndexFlatIP
索引向量數： 24
索引維度： 768


,FAISS編號,doc_id
0,0,doc_001
1,1,doc_002
2,2,doc_003
3,3,doc_004
4,4,doc_005
5,5,doc_006
6,6,doc_007
7,7,doc_008
8,8,doc_009
9,9,doc_010


## 12. 定義問題向量化與搜尋函式
Note: 計算問題與文件向量之間的相似度，預設取出最相關的 top_k=3 篇文章，並回傳文章內容和相似度分數跟來源資訊

In [ ]:
def embed_question(question):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("請輸入非空白問題。")
    query_input = QUERY_TEMPLATE.format(question=question.strip())
    if count_input_tokens(query_input) > TOKEN_LIMIT:
        raise ValueError("問題超過模型輸入長度上限，請縮短問題。")
    query_vector = model.encode(
        [query_input], prompt="", convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=False,
    )
    query_vector = np.ascontiguousarray(query_vector, dtype=np.float32)
    if query_vector.shape != (1, DIMENSION) or not np.isfinite(query_vector).all():
        raise ValueError("問題向量的維度或數值不正確。")
    if np.linalg.norm(query_vector) <= 0:
        raise ValueError("問題產生零向量。")
    faiss.normalize_L2(query_vector)
    return query_vector


def search_documents(question, top_k=3):
    if isinstance(top_k, bool) or not isinstance(top_k, (int, np.integer)) or top_k < 1:
        raise ValueError("top_k 必須是大於0的整數。")
    query_vector = embed_question(question)
    scores, positions = index.search(query_vector, min(int(top_k), index.ntotal))
    results = []
    for rank, (position, score) in enumerate(zip(positions[0], scores[0]), start=1):
        doc_id = index_doc_ids[int(position)]
        results.append({"rank": rank, "score": float(score), **documents_by_id[doc_id]})
    return results

## 13. 閱讀真正取回的內容

In [ ]:
QUESTION = "雞舍可以養哪些動物？"
TOP_K = 3
results = search_documents(QUESTION, TOP_K)
display(pd.DataFrame([
    {"排名": r["rank"], "doc_id": r["doc_id"], "標題": r["metadata"]["title"],
     "cosine_similarity": r["score"], "來源": r["metadata"]["source_url"]}
    for r in results
]))
for result in results:
    print(f"\n[{result['doc_id']}] {result['metadata']['title']} ({result['score']:.4f})")
    print(result["text"])
    print("來源：", result["metadata"]["source_url"] or "未提供")

,排名,doc_id,標題,cosine_similarity,來源
0,1,doc_007,雞舍與升級,0.803344,https://zh.stardewvalleywiki.com/鸡舍
1,2,doc_006,動物與畜牧,0.779638,https://zh.stardewvalleywiki.com/动物
2,3,doc_008,農舍與房屋升級,0.667681,https://zh.stardewvalleywiki.com/农舍



[doc_007] 雞舍與升級 (0.8033)
雞舍與升級

雞舍可以向木匠羅賓建造。最初的雞舍需要 4,000 金、300 木材與 100 石頭，容納 4 隻動物。升級成大雞舍需額外 10,000 金、400 木材及 150 石頭，可容納 8 隻，並附有孵化器。再升成高級雞舍需要 20,000 金、500 木材及 200 石頭，容量提高到 12 隻，並有自動餵食功能。
雞舍不只養雞；隨建築等級與解鎖進度，可養鴨、兔子、恐龍及特殊雞。基本雞舍的建造約需三天，升級約需兩天。大雞舍的孵化器能利用特定蛋孵出動物，高級雞舍則會從筒倉供應乾草。
讓動物出外吃草可節省飼料，但要注意天氣、季節，以及動物能否安全回到建築。
來源： https://zh.stardewvalleywiki.com/鸡舍

[doc_006] 動物與畜牧 (0.7796)
動物與畜牧

農場動物能提供雞蛋、牛奶、羊毛、松露等資源，也能作為賺錢方式。雞、鴨、兔子與恐龍等屬於雞舍動物；牛、山羊、綿羊與豬等則需要合適的畜棚。動物建築可向木匠羅賓購買或升級，農場動物通常可向瑪妮購買；部分動物也可透過孵化取得。
成年的雞舍動物有機會生產可撿取的產品；牛與山羊需要擠奶，綿羊可以剪毛，豬能在戶外尋找松露。動物需要餵食與照顧，互動、食物及活動條件也會影響心情與關係。
購買新動物前要確認建築等級與剩餘容量。動物產品可以直接出售，也能利用加工設備製成美乃滋、起司等工匠物品。
來源： https://zh.stardewvalleywiki.com/动物

[doc_008] 農舍與房屋升級 (0.6677)
農舍與房屋升級

農舍是玩家睡覺與生活的主要場所，可擺放家具、改變牆紙與地板，並透過羅賓進行升級。第一次升級需要 10,000 金與 450 木材，新增廚房、冰箱與更大的臥室，玩家可以利用爐灶烹飪。冰箱儲存的材料也能直接供廚房使用。
第二次升級需要 65,000 金與 100 硬木，擴大房內空間並加入兒童房；第三次升級需要 100,000 金，開放地下室。地下室中的木桶可讓部分飲品與起司陳釀，以提升品質。
房屋升級雖會增加內部面積，外部占地範圍不改變。若想開始烹飪或發展高價值加工，農舍升級是重要的投資方向。
來源： https://zh.stardewvalleywiki.com/农舍


## 14. 保存索引、文件、編號對照與設定

In [ ]:
EXPORT_DIR = Path(tempfile.mkdtemp(prefix="rag_index_"))
faiss.write_index(index, str(EXPORT_DIR / "index.faiss"))

for filename, data in [("documents.json", documents), ("index_doc_ids.json", index_doc_ids)]:
    (EXPORT_DIR / filename).write_text(
        json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8"
    )

config = {
    "format_version": "1.0",
    "created_at": datetime.now(ZoneInfo("Asia/Taipei")).isoformat(timespec="seconds"),
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "model_config_kwargs": {"vision_config": None, "audio_config": None},
    "dimension": DIMENSION,
    "normalize_embeddings": True,
    "index_type": "IndexFlatIP",
    "metric": "cosine_similarity_via_normalized_inner_product",
    "document_template": DOCUMENT_TEMPLATE,
    "query_template": QUERY_TEMPLATE,
    "encode_prompt": "",
    "max_seq_length": TOKEN_LIMIT,
    "document_count": len(documents),
    "document_unit": "one_document_one_vector",
    "document_text_includes_title": True,
    "package_versions": PACKAGE_VERSIONS,
    "build_device": DEVICE,
    "build_dtype": str(DTYPE),
    "source_archive": archive_name,
    "source_archive_sha256": SOURCE_ARCHIVE_SHA256,
    "file_sha256": {
        name: hashlib.sha256((EXPORT_DIR / name).read_bytes()).hexdigest()
        for name in ["index.faiss", "documents.json", "index_doc_ids.json"]
    },
}
(EXPORT_DIR / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("已保存：")
for path in sorted(EXPORT_DIR.iterdir()):
    print(path.name, f"({path.stat().st_size:,} bytes)")
print("\n模型版本及前綴已寫入 config.json。")

已保存：
config.json (1,390 bytes)
documents.json (22,836 bytes)
index.faiss (73,773 bytes)
index_doc_ids.json (314 bytes)

模型版本及前綴已寫入 config.json。


## 15. 驗證第二份 Colab 所需的資料

In [ ]:
loaded_config = json.loads((EXPORT_DIR / "config.json").read_text(encoding="utf-8"))
for name, expected_hash in loaded_config["file_sha256"].items():
    actual_hash = hashlib.sha256((EXPORT_DIR / name).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, f"檔案完整性檢查失敗：{name}"

loaded_index = faiss.read_index(str(EXPORT_DIR / "index.faiss"))
loaded_documents = json.loads((EXPORT_DIR / "documents.json").read_text(encoding="utf-8"))
loaded_ids = json.loads((EXPORT_DIR / "index_doc_ids.json").read_text(encoding="utf-8"))
assert loaded_index.ntotal == len(loaded_documents) == len(loaded_ids) == loaded_config["document_count"]
assert loaded_index.d == loaded_config["dimension"]
assert len(set(loaded_ids)) == len(loaded_ids)
assert set(loaded_ids) == {doc["doc_id"] for doc in loaded_documents}
assert loaded_documents == documents and loaded_ids == index_doc_ids

verification_query = embed_question(QUESTION)
k = min(TOP_K, index.ntotal)
before_scores, before_positions = index.search(verification_query, k)
after_scores, after_positions = loaded_index.search(verification_query, k)
np.testing.assert_array_equal(before_positions, after_positions)
np.testing.assert_allclose(before_scores, after_scores, atol=1e-6, rtol=1e-6)

all_scores = (embeddings @ verification_query[0]).ravel()
expected_top_scores = np.sort(all_scores)[::-1][:k]
np.testing.assert_allclose(after_scores[0], expected_top_scores, atol=1e-5, rtol=1e-5)
np.testing.assert_allclose(after_scores[0], all_scores[after_positions[0]], atol=1e-5, rtol=1e-5)
print("通過：檔案完整性、文件對照、索引維度、保存前後一致，以及精確搜尋檢查。")

通過：檔案完整性、文件對照、索引維度、保存前後一致，以及精確搜尋檢查。


## 16. 打包並下載知識庫

In [ ]:
from google.colab import files
EXPORT_ZIP = EXPORT_DIR.parent / f"{EXPORT_DIR.name}_download" / EXPORT_ZIP_NAME
EXPORT_ZIP.parent.mkdir(parents=True, exist_ok=True)
required_files = ["index.faiss", "documents.json", "index_doc_ids.json", "config.json"]
with zipfile.ZipFile(EXPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in required_files:
        archive.write(EXPORT_DIR / name, arcname=name)
with zipfile.ZipFile(EXPORT_ZIP) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(required_files)
print("下載檔名：", EXPORT_ZIP_NAME)
print("大小：", f"{EXPORT_ZIP.stat().st_size:,} bytes")
files.download(str(EXPORT_ZIP))

下載檔名： vector_embedding.zip
大小： 56,267 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>